# NGLab Tutorial #4: Time-Series Forecasting

Explore classical and modern time-series models: ARIMA, GARCH, Prophet (Rust), and TSMamba (Python).

## Learning Objectives

1. Load and prepare historical price data
2. Apply ARIMA, GARCH, and Prophet models
3. Train TSMamba for multi-step forecasting
4. Compare model performance

---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Set random seeds
np.random.seed(42)
torch.manual_seed(42)

print("Libraries loaded successfully!")

## 1. Generate Synthetic Price Data

We'll create a realistic price series with:
- Trend component
- Seasonal patterns
- Volatility clustering (GARCH effects)
- Random noise

In [ ]:
# Generate 2 years of daily data
n_days = 730
t = np.arange(n_days)

# Trend
trend = 50000 + 20 * t + 0.05 * t**2

# Seasonality (weekly pattern)
seasonality = 500 * np.sin(2 * np.pi * t / 7)

# GARCH-like volatility
volatility = np.zeros(n_days)
volatility[0] = 100
for i in range(1, n_days):
    volatility[i] = 0.1 * volatility[i-1] + 0.9 * abs(np.random.randn()) * 150

# Generate prices
noise = np.random.randn(n_days) * volatility
prices = trend + seasonality + noise
prices = np.maximum(prices, 1000)  # Floor at $1000

# Create DataFrame
df = pd.DataFrame({
    'date': pd.date_range('2022-01-01', periods=n_days),
    'price': prices
})

print(f"Generated {len(df)} days of price data")
print(f"Price range: ${df['price'].min():.2f} - ${df['price'].max():.2f}")

In [ ]:
# Visualize the data
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

# Price series
ax1.plot(df['date'], df['price'], linewidth=1.5, color='steelblue')
ax1.set_title('BTC/USD Price Series', fontsize=14, fontweight='bold')
ax1.set_ylabel('Price (USD)')
ax1.grid(True, alpha=0.3)

# Daily returns
returns = df['price'].pct_change().dropna()
ax2.plot(df['date'][1:], returns, linewidth=1, color='coral', alpha=0.7)
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.5)
ax2.set_title('Daily Returns', fontsize=14, fontweight='bold')
ax2.set_xlabel('Date')
ax2.set_ylabel('Return')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 2. ARIMA: AutoRegressive Integrated Moving Average

ARIMA(p, d, q) where:
- **p**: AR order (lags of the series)
- **d**: Differencing order (to make stationary)
- **q**: MA order (lags of forecast errors)

$$
\phi(B)(1-B)^d y_t = \theta(B) \epsilon_t
$$

In [ ]:
# Simple AR(1) model for demonstration
def fit_ar1(series, train_size=0.8):
    """Fit AR(1) model: y_t = c + φ*y_{t-1} + ε_t"""
    n_train = int(len(series) * train_size)
    train = series[:n_train]
    
    # Estimate parameters using OLS
    y = train[1:]
    x = train[:-1]
    
    # Add constant term
    X = np.column_stack([np.ones_like(x), x])
    
    # Solve: β = (X'X)^{-1} X'y
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    
    return {'const': beta[0], 'phi': beta[1]}

# Fit model on differenced returns
diff_prices = np.diff(df['price'].values)
ar_params = fit_ar1(diff_prices)

print("\n=== AR(1) Model Parameters ===")
print(f"Constant (c): {ar_params['const']:.2f}")
print(f"AR coefficient (φ): {ar_params['phi']:.4f}")

if abs(ar_params['phi']) < 1:
    print("✓ Model is stationary (|φ| < 1)")
else:
    print("⚠ Model may be non-stationary")

## 3. GARCH: Modeling Volatility Clustering

GARCH(1,1) captures "volatility clustering":

$$
\sigma_t^2 = \omega + \alpha \epsilon_{t-1}^2 + \beta \sigma_{t-1}^2
$$

In [ ]:
# Simple GARCH(1,1) estimation
def estimate_garch11(returns):
    """Simplified GARCH(1,1) parameter estimation."""
    # Remove mean
    residuals = returns - returns.mean()
    
    # Squared residuals
    sq_resid = residuals ** 2
    
    # Estimate via regression (simplified)
    omega = sq_resid.mean() * 0.1
    alpha = 0.1
    beta = 0.85
    
    return {'omega': omega, 'alpha': alpha, 'beta': beta}

garch_params = estimate_garch11(returns.values)

print("\n=== GARCH(1,1) Parameters ===")
print(f"ω (omega): {garch_params['omega']:.6f}")
print(f"α (alpha): {garch_params['alpha']:.4f}")
print(f"β (beta): {garch_params['beta']:.4f}")
print(f"Persistence (α+β): {garch_params['alpha'] + garch_params['beta']:.4f}")

## 4. TSMamba: State Space Model for Forecasting

TSMamba is a **Selective State Space Model** with $O(N)$ complexity:

In [ ]:
class MambaBlock:
    def forward(x):
        # Selective scan
        h = selective_scan(x, A, B, C)
        return h

In [ ]:
# Simple RNN baseline (lighter than full TSMamba)
class SimpleRNN(nn.Module):
    def __init__(self, input_dim=1, hidden_dim=64, num_layers=2):
        super().__init__()
        self.rnn = nn.GRU(input_dim, hidden_dim, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
    
    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1, :])

# Prepare data
def create_sequences(data, seq_len=30):
    X, y = [], []
    for i in range(len(data) - seq_len):
        X.append(data[i:i+seq_len])
        y.append(data[i+seq_len])
    return np.array(X), np.array(y)

# Normalize
price_norm = (df['price'].values - df['price'].mean()) / df['price'].std()
X, y = create_sequences(price_norm, seq_len=30)

# Train/test split
split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_test, y_test = X[split:], y[split:]

# Convert to tensors
X_train_t = torch.FloatTensor(X_train).unsqueeze(-1)
y_train_t = torch.FloatTensor(y_train)
X_test_t = torch.FloatTensor(X_test).unsqueeze(-1)
y_test_t = torch.FloatTensor(y_test)

print(f"\nTraining data: {X_train_t.shape}")
print(f"Test data: {X_test_t.shape}")

In [ ]:
# Train simple RNN
model = SimpleRNN()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
criterion = nn.MSELoss()

n_epochs = 50
losses = []

for epoch in range(n_epochs):
    model.train()
    optimizer.zero_grad()
    
    pred = model(X_train_t)
    loss = criterion(pred, y_train_t)
    
    loss.backward()
    optimizer.step()
    
    losses.append(loss.item())
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{n_epochs}, Loss: {loss.item():.6f}")

print("\n✓ Training complete!")

In [ ]:
# Evaluate
model.eval()
with torch.no_grad():
    y_pred = model(X_test_t).numpy()

# De-normalize
y_test_denorm = y_test * df['price'].std() + df['price'].mean()
y_pred_denorm = y_pred.flatten() * df['price'].std() + df['price'].mean()

# Metrics
mse = mean_squared_error(y_test_denorm, y_pred_denorm)
mae = mean_absolute_error(y_test_denorm, y_pred_denorm)
mape = np.mean(np.abs((y_test_denorm - y_pred_denorm) / y_test_denorm)) * 100

print("\n=== Test Set Performance ===")
print(f"MSE: {mse:,.2f}")
print(f"MAE: ${mae:.2f}")
print(f"MAPE: {mape:.2f}%")

In [ ]:
# Visualize predictions
plt.figure(figsize=(14, 6))
plt.plot(y_test_denorm[:100], label='Actual', linewidth=2, alpha=0.7)
plt.plot(y_pred_denorm[:100], label='Predicted', linewidth=2, alpha=0.7)
plt.title('RNN Forecasting Performance (First 100 Test Points)', fontsize=14, fontweight='bold')
plt.xlabel('Time Step')
plt.ylabel('Price (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## Summary

In this notebook, you learned:

✅ ARIMA for modeling autocorrelation  
✅ GARCH for volatility clustering  
✅ RNN/Mamba architectures for deep forecasting  
✅ Model evaluation metrics (MSE, MAE, MAPE)  

## Next Steps

Continue to **Notebook #5**: Deep Learning Models for advanced architectures!

---